# Tempos de Viagem por Caminhada

[descrição]



# Backend

In [9]:
# ============================= Standard library =============================
import datetime as dt
import os
import pathlib
import pickle
import subprocess
import tempfile
import xml.etree.ElementTree as ET
from pathlib import Path
from typing import Dict, Iterable, Mapping, Optional, Union, List, Callable

# ============================= Third-party libs ==============================
import geopandas as gpd
import h3
import matplotlib.pyplot as plt
import networkx as nx
import numpy as np
import osmnx as ox
import pandas as pd
import polars as pl
import polars_h3 as plh3
from pyrosm import OSM
import r5py
from shapely.geometry import Point
from tqdm.auto import tqdm


In [10]:
%matplotlib inline
%config InlineBackend.figure_format='retina'

pd.options.display.float_format = '{:,.2f}'.format

tqdm_bar_format = (
    "{desc:<12} {percentage:3.0f}%|{bar}| "
    "{n_fmt}/{total_fmt} • {rate_fmt} • {elapsed}<{remaining} {postfix}"
)

In [11]:
out_folder = os.environ.get('OUT_FOLDER')
out_folder = pathlib.Path(out_folder)

db_folder = os.environ.get('DB_FOLDER')
db_folder = pathlib.Path(db_folder)

## INPUTS

In [12]:
inpath = out_folder / 'A/pop_weighted_centroids.parquet'

centroids = gpd.read_parquet(inpath)

## osm.pbf

In [ ]:
#!/usr/bin/env python
# -*- coding: utf-8 -*-
"""
Goal
----
1) Read your dual edge_gdf with 'speed_kph' (kph).
2) Load an UNSIMPLIFIED OSMnx graph from a base .osm.pbf file.
3) Transfer speeds from dual edges -> OSMnx edges via sjoin_nearest.
4) Build {way_id -> maxspeed tags} and write a tiny .osc change file.
5) Apply it to the base .osm.pbf using 'osmium apply-changes' to create a new PBF.

Notes
-----
- Keep the OSMnx graph UNSIMPLIFIED to preserve OSM way IDs.
- We match by nearest line geometry in a projected CRS for sensible distances.
- We keep the code minimal: no heavy validation, just essentials.
"""
# ---------------------------- osmium helpers ------------------------------- #

def run_osmium(args: Iterable[str]) -> None:
    """Run an osmium CLI command, raising on failure."""
    # Ensure all args are strings
    str_args = [str(arg) for arg in args]
    proc = subprocess.run(str_args, capture_output=True, text=True, check=False)
    if proc.returncode != 0:
        msg = proc.stderr.strip() or proc.stdout.strip()
        raise RuntimeError(f"osmium error ({' '.join(str_args)}): {msg}")


def _write_id_file(ids: Iterable[int], path: Path) -> None:
    """Write 'w<ID>' lines for osmium getid."""
    with path.open("w", encoding="utf-8") as f:
        for wid in ids:
            f.write(f"w{int(wid)}\n")


def _extract_ways_xml(base_pbf: Path, ids_file: Path, out_xml: Path) -> None:
    """Get current versions of target ways as OSM XML."""
    run_osmium([
        "osmium", "getid",
        "-f", "osm",      # Output format OSM XML
        "-i", ids_file,  # Input ID file
        "-o", out_xml,   # Output XML file
        "-O",            # Allow overwrite
        base_pbf,        # Base PBF to read from
    ])


def apply_change_to_pbf(
    base_pbf: Union[str, Path],
    change_file: Union[str, Path],
    output_pbf: Union[str, Path],
) -> None:
    """Apply .osc to base PBF and write a new edited PBF."""
    run_osmium([
        "osmium", "apply-changes",
        "-o", output_pbf,  # Output PBF file
        "-O",             # Allow overwrite
        base_pbf,
        change_file,
    ])


# --------------------------- small general utils --------------------------- #

def _iter_osmids(osmid_val) -> Iterable[int]:
    """Yield int osmid(s) from scalar or iterable."""
    if osmid_val is None:
        return
    if isinstance(osmid_val, (list, tuple, set)):
        for v in osmid_val:
            if v is not None:
                yield int(v)
    else:
        try:
            yield int(osmid_val)
        except (ValueError, TypeError):
            return


def _fmt_speed_kph(val) -> Optional[str]:
    """Return clean numeric 'maxspeed' (km/h) or None."""
    if val is None:
        return None
    try:
        v = float(val)
    except (ValueError, TypeError):
        return None
    
    # OSM speeds must be positive, and 400 kph is a reasonable upper bound
    if not (0 < v < 400):
        return None
    
    # Return as integer string
    return str(int(round(v)))


# ------------------------ 1) load unsimplified OSMnx ----------------------- #

def load_unsimplified_edges(
    base_pbf: Union[str, Path],
) -> "nx.MultiDiGraph":
    """
    Load an UNSIMPLIFIED graph directly from the base PBF.
    """
    osm = OSM(base_pbf)
    return osm.get_network(network_type="driving")


# -- 3) project both edge sets; nearest-join speeds → OSMnx edges (midpoints)

def _project_for_distance(gdf: gpd.GeoDataFrame) -> gpd.GeoDataFrame:
    """Project to a local UTM CRS for distance calcs."""
    if gdf.crs is None:
        raise ValueError("GeoDataFrame lacks a CRS.")
    
    return gdf.to_crs(31983)


def transfer_speeds_via_nearest(
    dual_edges: gpd.GeoDataFrame,
    osm_edges: gpd.GeoDataFrame,
    *,
    max_dist_m: Optional[float] = None,
    dual_speed_col: str = "speed_kph",
) -> gpd.GeoDataFrame:
    """
    Nearest-join dual edge speeds onto OSM edges (line-to-line).
    Returns a copy of osm_edges with a new 'speed_kph' column.
    """
    if dual_speed_col not in dual_edges.columns:
        raise KeyError(f"Missing column in dual_edges: {dual_speed_col}")

    if dual_edges.crs is None or osm_edges.crs is None:
        raise ValueError("Both input GeoDataFrames must have a CRS.")

    # Project both to a metric CRS for meaningful distances
    dual_proj = _project_for_distance(dual_edges.copy())
    osm_proj = _project_for_distance(osm_edges.copy())

    # Nearest spatial join (line-to-line), keep distance in meters
    joined = gpd.sjoin_nearest(
        osm_proj[["geometry"]],  # Left GeoDataFrame (OSM edges)
        dual_proj[["geometry", dual_speed_col]], # Right (Dual edges)
        how="left",
        distance_col="dist_m",
    )

    # Resolve ties: smallest distance per left index
    joined = (
        joined.sort_values("dist_m")
              .groupby(level=0, sort=False)
              .first()
    )

    # Optional cutoff
    if max_dist_m is not None:
        joined.loc[joined["dist_m"] > max_dist_m, dual_speed_col] = pd.NA

    # Attach back to original osm_edges index and CRS
    out = osm_edges.copy()
    out["speed_kph"] = joined[dual_speed_col].reindex(out.index).values
    if out.crs is None:
        out = out.set_crs(osm_edges.crs, allow_override=True)

    return out


# --------- 4) build way_id -> tag map from osm_edges with speed_kph -------- #

def build_speed_map_from_osm_edges(
    osm_edges_with_speed: gpd.GeoDataFrame,
    *,
    osmid_col: str = "osmid",
    speed_col: str = "speed_kph",
) -> Dict[int, Dict[str, str]]:
    """
    Build {way_id: {'maxspeed': 'NN'}}. Handles osmid lists.
    """
    mapping: Dict[int, Dict[str, str]] = {}
    
    # Filter for rows that actually have a speed
    valid_rows = osm_edges_with_speed[osm_edges_with_speed[speed_col].notna()]
    
    for _, row in valid_rows.iterrows():
        osmids = list(_iter_osmids(row.get(osmid_col)))
        if not osmids:
            continue
        
        sp = _fmt_speed_kph(row.get(speed_col))
        if sp is None:
            continue
            
        for wid in osmids:
            # Note: Last one wins if a way ID appears multiple times
            mapping[wid] = {"maxspeed": sp}
            
    return mapping


# --------------- 5) craft minimal OsmChange with modified ways ------------- #

def _parse_ways(xml_path: Path) -> Dict[int, dict]:
    """Parse {id: {'version': v, 'nds': [...], 'tags':{...}}} from XML."""
    out: Dict[int, dict] = {}
    tree = ET.parse(xml_path)
    root = tree.getroot()
    for w in root.findall("way"):
        try:
            wid = int(w.attrib["id"])
            ver = int(w.attrib.get("version", "1"))
            nds = [int(nd.attrib["ref"]) for nd in w.findall("nd")]
            tags = {t.attrib["k"]: t.attrib["v"] for t in w.findall("tag")}
            out[wid] = {"version": ver, "nds": nds, "tags": tags}
        except (KeyError, ValueError) as e:
            # Keep this guardrail as bad XML is a real possibility
            print(f"Warning: Skipping malformed way in XML: {e}")
            continue
    return out


def build_change_file_for_maxspeeds(
    base_pbf: Union[str, Path],
    speed_map: Mapping[int, Mapping[str, str]],
    change_path: Union[str, Path],
) -> Path:
    """
    Produce a tiny .osc with <modify> entries for the listed way IDs.
    """
    base_pbf = Path(base_pbf)
    change_path = Path(change_path)
    
    if not speed_map:
        print("Speed map is empty. No change file will be generated.")
        return change_path

    with tempfile.TemporaryDirectory() as td:
        td_path = Path(td)
        ids_file = td_path / "ids.txt"
        ways_xml = td_path / "ways.xml"

        # 1. Write way IDs to a text file
        _write_id_file(speed_map.keys(), ids_file)
        
        # 2. Extract current version of these ways from base PBF
        _extract_ways_xml(base_pbf, ids_file, ways_xml)
        
        # 3. Parse the extracted XML
        base_ways = _parse_ways(ways_xml)

    # 4. Build the .osc
    osc = ET.Element("osmChange", version="0.6", generator="speed-patch")
    modify = ET.SubElement(osc, "modify")

    for wid, patch in speed_map.items():
        meta = base_ways.get(wid)
        if not meta:
            # Keep this guardrail as missing ways is a real possibility
            print(f"Warning: Way ID {wid} not found in base PBF. Skipping.")
            continue
        
        # Increment version number for the modification
        ver = int(meta["version"]) + 1
        w_elem = ET.SubElement(modify, "way", id=str(wid), version=str(ver))
        
        # Add node references
        for nref in meta["nds"]:
            ET.SubElement(w_elem, "nd", ref=str(nref))
            
        # Add tags, applying our patch
        tags = dict(meta["tags"])
        tags.update(patch)  # This overwrites/adds 'maxspeed'
        
        for k, v in sorted(tags.items()):
            ET.SubElement(w_elem, "tag", k=str(k), v=str(v))

    # 5. Write .osc file to disk
    ET.ElementTree(osc).write(
        change_path, encoding="utf-8", xml_declaration=True
    )
    print(f"Wrote {len(speed_map)} way updates to change file: {change_path}")
    return change_path


In [ ]:

# ========================================================================= #
#                               MAIN SCRIPT
# ========================================================================= #

# --- 1. DEFINE USER INPUTS (Must be set by user) ---
# Path to your source .osm.pbf file
base_pbf = r"C:\Users\brand\OneDrive\Documentos\Coppe\thesis\outputs\B\bh.osm.pbf"

# Placeholder: Load your 'edge_gdf' (dual graph)
OUT_GDUAL_PICKLE = out_folder / "B/cityseer/G_dual_car_speeds.gpickle"
with open(OUT_GDUAL_PICKLE, "rb") as f:
    edge_gdf = pickle.load(f)
edge_gdf['speed_kph'] = edge_gdf['speed_mps'] * 3.6
edge_gdf = edge_gdf.to_crs(31983)

# Path to your output folder
out_folder = Path(out_folder / "B")
out_folder.mkdir(parents=True, exist_ok=True)

# --- 3. Load UNSIMPLIFIED graph from base_pbf ---
osm_edges = load_unsimplified_edges(base_pbf)

# Ensure osm_edges has the same base CRS as dual_edges before projection
if osm_edges.crs != edge_gdf.crs:
        osm_edges = osm_edges.to_crs(edge_gdf.crs)

osm_edges_with_speed = transfer_speeds_via_nearest(
    dual_edges=edge_gdf,
    osm_edges=osm_edges,
    max_dist_m=50.0,  # Set a reasonable max distance (e.g., 50 meters)
    dual_speed_col="speed_kph",
)

# --- 5. Build {way_id -> {'maxspeed': 'NN'}} map ---
speed_map = build_speed_map_from_osm_edges(
    osm_edges_with_speed,
    osmid_col="id",
    speed_col="speed_kph",
)
print(f"Found {len(speed_map)} ways to update.")

# --- 6. Create .osc and apply to base .osm.pbf ---
if speed_map:
    change_osc = out_folder / "maxspeed_edits.osc"
    edited_pbf = out_folder / "output_with_new_speeds.osm.pbf"

    build_change_file_for_maxspeeds(base_pbf, speed_map, change_osc)
    
    print(f"Applying changes to create {edited_pbf}...")
    apply_change_to_pbf(base_pbf, change_osc, edited_pbf)

    print(f"Done. Wrote new PBF to: {edited_pbf}")
else:
    print("No speed updates to apply.")



In [ ]:
print("edge_gdf CRS:", edge_gdf.crs)
print("osm_edges CRS:", osm_edges.crs)
print("non-null matches in osm_edges_with_speed:",
      int(osm_edges_with_speed["speed_kph"].notna().sum()))
print("columns in osm_edges_with_speed:", list(osm_edges_with_speed.columns))


## Routing

In [19]:
ORIGINS_DESTINATIONS = (
    centroids
    .rename(columns={'hex_id': 'id'})
    .to_crs(31983) # just in case...
    .reindex(columns=['id', 'aperture', 'geometry'])
    .to_crs(4326)
    .query('aperture == 10')
    )

In [ ]:
with tqdm(
    ORIGINS_DESTINATIONS.groupby("aperture"),
    desc="Apertures",
    bar_format=tqdm_bar_format,
    colour="#AA4499"
    ) as outer_bar:
    for aperture, data in outer_bar:
        outer_bar.set_postfix_str(f"Resolution {aperture}")

        outpath = out_folder / "B"
        aperture_dir = outpath / f"travel_times/car"
        aperture_dir.mkdir(parents=True, exist_ok=True)

        transport_network = r5py.TransportNetwork(
            edited_pbf,
            elevation_model=out_folder.parent / 'A/dem_bh.tiff',
        )

        travel_times = r5py.TravelTimeMatrix(
            transport_network,
            origins=data,
            max_time=dt.timedelta(minutes=120),
            transport_modes=[
                r5py.TransportMode.CAR,
            ],
            snap_to_network=np.ceil(
                h3.average_hexagon_edge_length(
                    aperture,
                    unit="m"
                )
            ),
        )

        parquet_file = aperture_dir / f"aperture_{aperture}.parquet"
        travel_times.to_parquet(parquet_file, index=False)


In [12]:
out_folder = Path(out_folder / "B")
outpath = out_folder / "B"
aperture_dir = outpath / f"travel_times/car"
aperture_dir.mkdir(parents=True, exist_ok=True)
parquet_file = aperture_dir / f"aperture_{9}.parquet"
travel_times = pd.read_parquet(parquet_file)

travel_times = pl.from_dataframe(travel_times.dropna())

travel_times

from_id,to_id,travel_time,__index_level_0__
str,str,f64,i64
"""89a881345b7ffff""","""89a881345b7ffff""",0.0,0
"""89a881345b7ffff""","""89a88136103ffff""",13.0,1
"""89a881345b7ffff""","""89a88136107ffff""",13.0,2
"""89a881345b7ffff""","""89a8813610bffff""",13.0,3
"""89a881345b7ffff""","""89a8813610fffff""",13.0,4
…,…,…,…
"""89a88cdb6dbffff""","""89a88cdb6cbffff""",3.0,8508884
"""89a88cdb6dbffff""","""89a88cdb6cfffff""",4.0,8508885
"""89a88cdb6dbffff""","""89a88cdb6d3ffff""",1.0,8508886


In [5]:
import polars as pl
import polars_h3 as plh3

# 1) Keep only needed columns and force join keys to Utf8 (match maps)
NUM_COLS = ["travel_time"]
df = (
    travel_times
    .select(["from_id", "to_id", *NUM_COLS])
    .with_columns(
        from_id=plh3.str_to_int("from_id"),
        to_id=plh3.str_to_int("to_id"),
    )
)

# 2) Build unique parent→children maps at res=10 (Utf8 throughout)
def children_map(df: pl.DataFrame, col: str) -> pl.DataFrame:
    parents = (
        df.select(pl.col(col).drop_nulls().unique().alias(col))
    )
    return (
        parents
        .with_columns(
            children=plh3.cell_to_children(col, 10)
            )
        .explode("children")
        .rename({"children": f"{col}_child"})
)

from_map = children_map(df, "from_id")
to_map   = children_map(df, "to_id")


In [6]:

# (optional) quick diagnostic of fan-out
# print(from_map.group_by("from_id").len().select(pl.col("len").mean()))
# print(to_map.group_by("to_id").len().select(pl.col("len").mean()))

# 3) Expand both sides to res=10, then aggregate by child pair (median)
result = (
    df.lazy()
      .join(from_map.lazy(), on="from_id", how="inner")
      .with_columns(pl.col("from_id_child").alias("from_id"))
      .drop("from_id_child")
      .join(to_map.lazy(), on="to_id", how="inner")
      .with_columns(pl.col("to_id_child").alias("to_id"))
      .drop("to_id_child")
      .group_by(["from_id", "to_id"])
      .median()
      .collect(engine="streaming")
)

# `result` now has (from_id_res10, to_id_res10) with median travel_time.


In [7]:
result

from_id,to_id,travel_time
u64,u64,f64
624461114699546623,624461115347501055,11.0
624461114699448319,624461115371978751,20.0
624461114699644927,624461144805670911,55.0
624461114699481087,624461144809603071,52.0
624461114699513855,624461144808062975,56.0
…,…,…
624461114699644927,624461115318435839,2.0
624461114699513855,624461144857378815,52.0
624461114699481087,624461147041300479,40.0


In [22]:
import numpy as np
import pandas as pd
import geopandas as gpd
from shapely.geometry import box
from pathlib import Path
from scipy.spatial.distance import cdist # For travel time matrix
from tobler.util import h3fy

# Set a random seed for reproducible toy data
np.random.seed(42)

# --- Configuration ---
OUT_DIR = Path("tests/data")
CLASSES = ["residential", "commercial", "industrial"]
H3_RESOLUTION = 10 # H3 resolution level (adjust for cell count)

# Approximate Bounding Box for Belo Horizonte CBD
#BH_CBD_LAT_MIN, BH_CBD_LON_MIN = -19.935, -43.955
#BH_CBD_LAT_MAX, BH_CBD_LON_MAX = -19.910, -43.925

BH_CBD_LAT_MIN, BH_CBD_LON_MIN = -20.0596990538, -44.0633406009
BH_CBD_LAT_MAX, BH_CBD_LON_MAX = -19.7765437127, -43.8565829615

BH_BBOX_WGS84 = box(BH_CBD_LON_MIN, BH_CBD_LAT_MIN, BH_CBD_LON_MAX, BH_CBD_LAT_MAX)
# Use a projected CRS appropriate for BH (e.g., SIRGAS 2000 / UTM zone 23S)
CRS_PROJECTED = "EPSG:31983"
CRS_WGS84 = "EPSG:4326"

# Travel time assumptions (meters per minute)
WALK_SPEED_MPM = 80.0  # ~4.8 km/h
CAR_SPEED_MPM = 400.0  # ~24 km/h (Lower for CBD)
BUS_SPEED_MPM = 250.0  # ~15 km/h (Lower for CBD, includes stops)
INTRA_CELL_DIST_M = 50.0 # Assumed avg. distance for intra-cell travel

# --- Setup ---
OUT_DIR.mkdir(parents=True, exist_ok=True)

print(f"Generating H3 grid (res {H3_RESOLUTION}) for BH CBD bounding box...")

# --- 1. Create H3 GeoDataFrame (Cells) ---
# Create GeoDataFrame for the bounding box
bbox_gdf = gpd.GeoDataFrame([1], geometry=[BH_BBOX_WGS84], crs=CRS_WGS84)

# Generate H3 grid using toblers
# Note: Ensure toblers/h3-py are installed
h3_grid_gdf = h3fy(bbox_gdf, resolution=H3_RESOLUTION)

# Project to UTM for distance calculations in meters
h3_grid_gdf = h3_grid_gdf.to_crs(CRS_PROJECTED)

NUM_CELLS = len(h3_grid_gdf)
ids = [f"c{i}" for i in range(NUM_CELLS)] # Simple sequential IDs
h3_ids_map = dict(zip(h3_grid_gdf.index, ids)) # Map H3 index to simple ID
h3_grid_gdf['cell_id'] = ids
h3_grid_gdf.set_index('cell_id', inplace=True)
h3_grid_gdf = h3_grid_gdf[['geometry']] # Keep only geometry

print(f"Generated {NUM_CELLS} cells.")

Generating H3 grid (res 10) for BH CBD bounding box...
Generated 41889 cells.


In [8]:
df

from_id,to_id,travel_time
u64,u64,f64
619957515072307199,619957515072307199,0.0
619957515072307199,619957515530272767,13.0
619957515072307199,619957515530534911,13.0
619957515072307199,619957515530797055,13.0
619957515072307199,619957515531059199,13.0
…,…,…
619958315834408959,619958315833360383,3.0
619958315834408959,619958315833622527,4.0
619958315834408959,619958315833884671,1.0
